In [1]:
import matplotlib as mpl
import matplotlib.gridspec as gridspec
import numpy as np
import xarray as xr
from matplotlib import pyplot as plt
from matplotlib import rcParams, colors
from matplotlib import ticker as mticker

rcParams.update({'figure.autolayout': False})
import os

import glob
import pandas as pd
import pickle


import dask
from dask.distributed import Client, LocalCluster
from dask import delayed
import dask.array as da
from dask_jobqueue import PBSCluster
import xarray as xr
from concurrent.futures import ProcessPoolExecutor
import calendar

mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

In [2]:
# Setup your PBSCluster
cluster = PBSCluster(
    cores=10,  # Total number of cores across all processes
    memory='300GB',  # Memory per worker
    processes=10,  # Number of processes per worker (1 process per core)
    queue='casper',  # The type of queue to utilize
    local_directory='/glade/derecho/scratch/adixit/tmp/',  # Use your local directory
    resource_spec='select=1:ncpus=10:mem=300GB',  # Resources per worker
    account='UCOR0077',  # Project ID
    walltime='1:00:00',  # Wall time
    #interface='ib0',  # Interface to use (if needed)
)

cluster.scale(jobs=1)  # This will create 200 workers
client = Client(cluster)

In [4]:
client

<Client: 'tcp://128.117.208.178:33957' processes=10 threads=10, memory=279.40 GiB>

In [5]:
def subset_india(ds):
    if 'lat' in ds.dims:
        ds = ds.rename({'lat':'latitude','lon':'longitude'})
    ds = ds.sel(latitude=slice(40, 0), longitude=slice(50, 100))
    ds = ds.sel(time=~((ds.time.dt.month == 2) & (ds.time.dt.day == 29)))
    return ds

def fourier(period, k, n):
    # Get periods for each fourier
    p = [x/period for x in range(1,k+1)]
    # Create array of ones
    result = np.ones( (n, 2*k+1) )
    # Fill in array with fourier series
    for i in range(k):
        result[:,(2*i)] = [np.sin(2*np.pi*x*p[i]) for x in range(1,n+1)]
        result[:,(2*i+1)] = [np.cos(2*np.pi*x*p[i]) for x in range(1,n+1)]
    return result

def seasonality_removal_vals(data, k):
    n=len(data)
    # Create fourier series to regress
    fX = fourier(365, k=k, n=n)
    B = np.linalg.inv(fX.T.dot(fX)).dot(fX.T)
    # reshape data to apply worker function to each gridcell
    data_reshape = data.reshape((data.shape[0], data.shape[1] * data.shape[2]))
    result_reshape = np.apply_along_axis(seasonality_removal_worker, 0, data_reshape, I = B, X = fX)
    result = result_reshape.reshape((data.shape[0], data.shape[1], data.shape[2]))
    return result

def seasonality_removal_worker(data, I, X):    
    # Regress fourier series on data
    B = I.dot(data)
    y_pred = X.dot(B)
    return data - y_pred

def seasonality_removal(data, k):
    n=len(data)
    # Create fourier series to regress
    fX = fourier(365, k=k, n=n)
    B = np.linalg.inv(fX.T.dot(fX)).dot(fX.T)
    # Stack data and apply worker function to each gridcell
    data_stack = data.stack(gridcell=('lat', 'lon'))
    result = data_stack.groupby('gridcell').map(seasonality_removal_worker, I = B, X = fX).unstack('gridcell')
    return result

def anom(x):
    n = x.sizes["time"]
    X = fourier(365, k=3, n=n)
    I = np.linalg.pinv(X)

    result = xr.apply_ufunc(
        seasonality_removal_worker,
        x.chunk({"time": -1,'latitude':10,'longitude':10}),
        input_core_dims=[["time"]],
        output_core_dims=[["time"]],
        vectorize=True,
        dask="parallelized",
        kwargs={"I": I, "X": X},
        output_dtypes=[x.dtype]
    )

    #result = result.transpose(['time','latitude','longitude'])
    return result

def z(x):
    b = x.sel(time=slice("1950", "2020"))
    mean = b.groupby("time.dayofyear").mean("time")
    std = b.groupby("time.dayofyear").std("time")
    result = x.groupby("time.dayofyear") - mean
    result = result.groupby("time.dayofyear") / std
    return result

def leap_adjusted_doy(time):
    doy = time.dt.dayofyear
    is_leap = time.dt.is_leap_year
    after_feb = time.dt.month > 2
    return doy - (is_leap & after_feb).astype(int)

In [6]:
msnswrf = xr.open_mfdataset('/glade/derecho/scratch/adixit/India_Heatwave/DATA/era5_data/msnswrf/*.nc', preprocess=subset_india)['MSNSWRF']
msnlwrf = xr.open_mfdataset('/glade/derecho/scratch/adixit/India_Heatwave/DATA/era5_data/msnlwrf/*.nc', preprocess=subset_india)['MSNLWRF']
mer = xr.open_mfdataset('/glade/derecho/scratch/adixit/India_Heatwave/DATA/era5_data/mer/*.nc', preprocess=subset_india)['MER']
mper = xr.open_mfdataset('/glade/derecho/scratch/adixit/India_Heatwave/DATA/era5_data/mper/*.nc', preprocess=subset_india)['MPER']
tmax = xr.open_mfdataset('/glade/derecho/scratch/adixit/India_Heatwave/DATA/era5_data/VAR_2T/*.nc', preprocess=subset_india)['VAR_2T']

ds_t2 = xr.open_mfdataset('/glade/derecho/scratch/adixit/India_Heatwave/dynamic_adjustment/AMJJAS_anom/VAR_2T_ERA5_India_AMJJAS_anom.nc', chunks='auto', preprocess=subset_india)
ds_da = xr.open_mfdataset('/glade/derecho/scratch/adixit/India_Heatwave/dynamic_adjustment/L1/dyn_adj/MSL_1950/dyn_adj_ERA5_India_50_34_20_74.nc', chunks='auto', preprocess=subset_india)
ds_sm = xr.open_mfdataset('/glade/derecho/scratch/adixit/India_Heatwave/dynamic_adjustment/AMJJAS_anom/SWVL1_ERA5_India_AMJJAS_anom.nc', chunks='auto', preprocess=subset_india)
#ds_t2_masked = ds_t2.sel(time=ds_da.time, lat=ds_da.lat, lon=ds_da.lon)
ds_diff = dask.delayed(ds_t2['anom'] - ds_da['dynamic'])
ds_diff = ds_diff.compute()

In [7]:
msnswrf = dask.compute(msnswrf)[0]
msnlwrf = dask.compute(msnlwrf)[0]
mer = dask.compute(mer)[0]
mper = dask.compute(mper)[0]
tmax = dask.compute(tmax)[0]
ds_t2 = dask.compute(ds_t2['anom'])[0]
ds_da = dask.compute(ds_da['dynamic'])[0]
ds_sm = dask.compute(ds_sm)[0]
Tres = ds_diff.compute()

In [8]:
n = msnlwrf.isnull().sum(("latitude", "longitude"))
i = int(n.argmax("time").compute())

print(i, msnlwrf.time.isel(time=i).values, int(n.isel(time=i).compute()))

0 1950-01-01T00:00:00.000000000 0


In [9]:
Lv = 2.45e6
Rn = msnswrf + msnlwrf
H  = Rn - (-mer * Lv)
Hp = Rn - (-mper * Lv)
tmax = tmax

T, H, Hp = map(anom, [tmax, H, Hp])

T = T.compute()
H = H.compute()
Hp = Hp.compute()

T, H, Hp = map(z, [T, H, Hp])

p = T * (H - Hp)
P = xr.corr(H, T, dim="time") - xr.corr(Hp, T, dim="time")

/glade/work/adixit/conda-envs/dask/lib/python3.10/site-packages/distributed/client.py:3164: UserWarning: Sending large graph of size 3.74 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(
/glade/work/adixit/conda-envs/dask/lib/python3.10/site-packages/distributed/client.py:3164: UserWarning: Sending large graph of size 3.30 GiB.
This may cause some slowdown.
Consider scattering data ahead of time and using futures.
  warnings.warn(


In [10]:
Tres = z(Tres)
Tres = Tres#.sel(time=Tres.time.dt.year==2024)
Hres = H.sel(time=Tres.time)
Hpres = Hp.sel(time=Tres.time)

In [11]:
pres = Tres * (Hres - Hpres)
Pres = xr.corr(Hres, Tres, dim="time") - xr.corr(Hpres, Tres, dim="time")

In [12]:
sm_anom = z(ds_sm['anom']).compute()
mask = sm_anom.time.dt.month.isin([5, 6, 7, 8]) & (sm_anom.time.dt.year == 2024)
sm_anom_nci_2024 = sm_anom.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)).mean(["latitude", "longitude"])

mask = ds_t2.time.dt.month.isin([5, 6, 7, 8]) & (ds_t2.time.dt.year == 2024)
t2_anom_nci_2024 = ds_t2.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)).mean(["latitude", "longitude"])

mask = ds_da.time.dt.month.isin([5, 6, 7, 8]) & (ds_da.time.dt.year == 2024)
t2_dynamic_nci_2024 = ds_da.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)).mean(["latitude", "longitude"])

mask = ds_diff.time.dt.month.isin([5, 6, 7, 8]) & (ds_diff.time.dt.year == 2024)
t2_res_nci_2024 = ds_diff.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)).mean(["latitude", "longitude"])

mask = pres.time.dt.month.isin([5, 6, 7, 8]) & (pres.time.dt.year == 2024)
pres_nci_2024 = pres.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)).mean(["latitude", "longitude"])

mask = p.time.dt.month.isin([5, 6, 7, 8]) & (p.time.dt.year == 2024)
p_nci_2024 = p.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)).mean(["latitude", "longitude"])

mask = T.time.dt.month.isin([5, 6, 7, 8]) & (T.time.dt.year == 2024)
T_nci_2024 = T.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)).mean(["latitude", "longitude"])

mask = Tres.time.dt.month.isin([5, 6, 7, 8]) & (Tres.time.dt.year == 2024)
Tres_nci_2024 = Tres.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)).mean(["latitude", "longitude"])

mask = H.time.dt.month.isin([5, 6, 7, 8]) & (H.time.dt.year == 2024)
H_nci_2024 = H.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)).mean(["latitude", "longitude"])

mask = Hres.time.dt.month.isin([5, 6, 7, 8]) & (Hres.time.dt.year == 2024)
Hres_nci_2024 = Hres.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)).mean(["latitude", "longitude"])

mask = Hp.time.dt.month.isin([5, 6, 7, 8]) & (Hp.time.dt.year == 2024)
Hp_nci_2024 = Hp.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)).mean(["latitude", "longitude"])

mask = Hpres.time.dt.month.isin([5, 6, 7, 8]) & (Hpres.time.dt.year == 2024)
Hpres_nci_2024 = Hpres.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)).mean(["latitude", "longitude"])

In [13]:
df2024 = np.transpose(pd.DataFrame([sm_anom_nci_2024.values,p_nci_2024.values,T_nci_2024.values,H_nci_2024.values,Hp_nci_2024.values,pres_nci_2024.values,Tres_nci_2024.values,Hres_nci_2024.values,
                                Hpres_nci_2024.values,t2_anom_nci_2024.values,t2_dynamic_nci_2024.values,t2_res_nci_2024.values]))
df2024.columns = ['SM_NCI','P_NCI','T_NCI','H_NCI','Hp_NCI','Pres_NCI','Tres_NCI','Hres_NCI','Hpres_NCI','T2_anom_NCI','T2_dyn_NCI','T2_res_NCI']
df2024

,SM_NCI,P_NCI,T_NCI,H_NCI,Hp_NCI,Pres_NCI,Tres_NCI,Hres_NCI,Hpres_NCI,T2_anom_NCI,T2_dyn_NCI,T2_res_NCI
0,-0.141491,0.473544,-0.488472,-0.684983,-0.618786,0.604818,-0.851553,-0.684983,-0.618786,-1.606956,0.231294,-1.838250
1,-0.103752,1.025383,-0.745926,-0.372193,-0.091091,0.524855,-0.956810,-0.372193,-0.091091,-2.365659,0.342019,-2.707678
2,-0.096265,0.647912,-0.006591,-0.121615,0.336701,1.487904,0.092408,-0.121615,0.336701,-0.433702,-1.059856,0.626154
3,-0.089578,0.118913,0.161669,-0.992196,-0.019988,0.200009,0.270963,-0.992196,-0.019988,0.033105,-0.935676,0.968781
4,-0.145817,0.349946,0.532096,-0.329564,-0.282378,0.462365,0.373677,-0.329564,-0.282378,1.213242,0.087139,1.126104
...,...,...,...,...,...,...,...,...,...,...,...,...
118,0.459102,0.643133,0.235284,-0.629793,-0.322314,0.381737,0.700562,-0.629793,-0.322314,0.666987,-0.447612,1.114599
119,0.333123,0.639208,0.427924,-0.118954,-0.073680,0.472270,0.648581,-0.118954,-0.073680,1.180612,0.178733,1.001878
120,0.271700,0.525046,0.629914,0.262204,0.304293,0.498127,0.695197,0.262204,0.304293,1.547100,0.209528,1.337572
121,0.152211,0.424839,1.048199,0.661566,0.544480,0.373810,1.037234,0.661566,0.544480,2.305509,0.145410,2.160099


In [14]:
df2024.to_csv('Land-surface-amplification_2024_new.csv')

In [18]:
percentile = 10

sm_anom = z(ds_sm['anom']).compute()
sm_anom = sm_anom.assign_coords(doy_adj=leap_adjusted_doy(sm_anom.time)) 
mask = sm_anom.time.dt.month.isin([5, 6, 7, 8]) & sm_anom.time.dt.year.isin(range(1950, 2024))
sm_anom_nci_xth = sm_anom.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)) \
    .mean(["latitude", "longitude"]) \
    .groupby("doy_adj") \
    .quantile(percentile / 100, dim="time")

ds_t2 = sm_anom.assign_coords(doy_adj=leap_adjusted_doy(ds_t2.time)) 
mask = ds_t2.time.dt.month.isin([5, 6, 7, 8]) & ds_t2.time.dt.year.isin(range(1950, 2024))
t2_anom_nci_xth = ds_t2.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)) \
    .mean(["latitude", "longitude"]) \
    .groupby("doy_adj") \
    .quantile(percentile / 100, dim="time")

ds_da = sm_anom.assign_coords(doy_adj=leap_adjusted_doy(ds_da.time)) 
mask = ds_da.time.dt.month.isin([5, 6, 7, 8]) & ds_da.time.dt.year.isin(range(1950, 2024))
t2_dynamic_nci_xth = ds_da.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)) \
    .mean(["latitude", "longitude"]) \
    .groupby("doy_adj") \
    .quantile(percentile / 100, dim="time")

ds_diff = sm_anom.assign_coords(doy_adj=leap_adjusted_doy(ds_diff.time)) 
mask = ds_diff.time.dt.month.isin([5, 6, 7, 8]) & ds_diff.time.dt.year.isin(range(1950, 2024))
t2_res_nci_xth = ds_diff.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)) \
    .mean(["latitude", "longitude"]) \
    .groupby("doy_adj") \
    .quantile(percentile / 100, dim="time")

pres = sm_anom.assign_coords(doy_adj=leap_adjusted_doy(pres.time)) 
mask = pres.time.dt.month.isin([5, 6, 7, 8]) & pres.time.dt.year.isin(range(1950, 2024))
pres_nci_xth = pres.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)) \
    .mean(["latitude", "longitude"]) \
    .groupby("doy_adj") \
    .quantile(percentile / 100, dim="time")

p = p.assign_coords(doy_adj=leap_adjusted_doy(p.time)) 
mask = p.time.dt.month.isin([5, 6, 7, 8]) & p.time.dt.year.isin(range(1950, 2024))
p_nci_xth = p.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)) \
    .mean(["latitude", "longitude"]) \
    .groupby("doy_adj") \
    .quantile(percentile / 100, dim="time")

T = T.assign_coords(doy_adj=leap_adjusted_doy(T.time)) 
mask = T.time.dt.month.isin([5, 6, 7, 8]) & T.time.dt.year.isin(range(1950, 2024))
T_nci_xth = T.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)) \
    .mean(["latitude", "longitude"]) \
    .groupby("doy_adj") \
    .quantile(percentile / 100, dim="time")

Tres = Tres.assign_coords(doy_adj=leap_adjusted_doy(Tres.time)) 
mask = Tres.time.dt.month.isin([5, 6, 7, 8]) & Tres.time.dt.year.isin(range(1950, 2024))
Tres_nci_xth = Tres.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)) \
    .mean(["latitude", "longitude"]) \
    .groupby("doy_adj") \
    .quantile(percentile / 100, dim="time")

H = H.assign_coords(doy_adj=leap_adjusted_doy(H.time)) 
mask = H.time.dt.month.isin([5, 6, 7, 8]) & H.time.dt.year.isin(range(1950, 2024))
H_nci_xth = H.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)) \
    .mean(["latitude", "longitude"]) \
    .groupby("doy_adj") \
    .quantile(percentile / 100, dim="time")

Hres = Hres.assign_coords(doy_adj=leap_adjusted_doy(Hres.time)) 
mask = Hres.time.dt.month.isin([5, 6, 7, 8]) & Hres.time.dt.year.isin(range(1950, 2024))
Hres_nci_xth = Hres.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)) \
    .mean(["latitude", "longitude"]) \
    .groupby("doy_adj") \
    .quantile(percentile / 100, dim="time")

Hp = Hp.assign_coords(doy_adj=leap_adjusted_doy(Hp.time)) 
mask = Hp.time.dt.month.isin([5, 6, 7, 8]) & Hp.time.dt.year.isin(range(1950, 2024))
Hp_nci_xth = Hp.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)) \
    .mean(["latitude", "longitude"]) \
    .groupby("doy_adj") \
    .quantile(percentile / 100, dim="time")

Hpres = Hpres.assign_coords(doy_adj=leap_adjusted_doy(Hpres.time))
mask = Hpres.time.dt.month.isin([5, 6, 7, 8]) & Hpres.time.dt.year.isin(range(1950, 2024))
Hpres_nci_xth = Hpres.sel(time=mask, longitude=slice(75, 85), latitude=slice(30, 22)) \
    .mean(["latitude", "longitude"]) \
    .groupby("doy_adj") \
    .quantile(percentile / 100, dim="time")

In [19]:
df90 = np.transpose(pd.DataFrame([sm_anom_nci_xth.values,p_nci_xth.values,T_nci_xth.values,H_nci_xth.values,Hp_nci_xth.values,pres_nci_xth.values,Tres_nci_xth.values,Hres_nci_xth.values,
                                Hpres_nci_xth.values,t2_anom_nci_xth.values,t2_dynamic_nci_xth.values,t2_res_nci_xth.values]))
df90.columns = ['SM_NCI','P_NCI','T_NCI','H_NCI','Hp_NCI','Pres_NCI','Tres_NCI','Hres_NCI','Hpres_NCI','T2_anom_NCI','T2_dyn_NCI','T2_res_NCI']
df90

,SM_NCI,P_NCI,T_NCI,H_NCI,Hp_NCI,Pres_NCI,Tres_NCI,Hres_NCI,Hpres_NCI,T2_anom_NCI,T2_dyn_NCI,T2_res_NCI
0,-0.567174,0.081681,-1.435364,-0.658785,-0.834581,-0.567174,-1.154635,-0.658785,-0.834581,-0.567174,-0.567174,-0.567174
1,-0.540729,0.067687,-1.234009,-0.755643,-0.776918,-0.540729,-0.906175,-0.755643,-0.776918,-0.540729,-0.540729,-0.540729
2,-0.562137,0.099227,-1.124184,-0.687391,-0.775753,-0.562137,-1.185969,-0.687391,-0.775753,-0.562137,-0.562137,-0.562137
3,-0.550779,-0.006985,-1.308741,-0.597826,-0.823129,-0.550779,-0.995043,-0.597826,-0.823129,-0.550779,-0.550779,-0.550779
4,-0.549094,0.094630,-1.352009,-0.667318,-0.787989,-0.549094,-0.927248,-0.667318,-0.787989,-0.549094,-0.549094,-0.549094
...,...,...,...,...,...,...,...,...,...,...,...,...
118,-1.008437,0.124238,-0.810345,-0.742853,-0.899141,-1.008437,-0.786916,-0.742853,-0.899141,-1.008437,-1.008437,-1.008437
119,-1.161309,0.145885,-0.875040,-0.746881,-0.745122,-1.161309,-0.676246,-0.746881,-0.745122,-1.161309,-1.161309,-1.161309
120,-1.108835,0.101209,-0.819245,-0.722022,-0.803077,-1.108835,-0.738113,-0.722022,-0.803077,-1.108835,-1.108835,-1.108835
121,-0.972467,0.157767,-0.776672,-0.645045,-0.591614,-0.972467,-0.733156,-0.645045,-0.591614,-0.972467,-0.972467,-0.972467


In [20]:
df90.to_csv('Land-surface-amplification_10pct_new.csv')